# Your Evals Are Only As Honest As The Data Behind Them

*Meet Halcyon: the synthetic payments API, documentation corpus, and support ticket set that every post in this series runs on, and the design decisions that let them catch anything.*

**Read this if**

- The real tickets behind a product cannot go into a notebook or a shared test suite. The eval data has to be made up, and it is not obvious what it must keep.
- This is the first post in the series. It sets up the company, the assistant, the documents, the tickets, and the setup cell every later post starts with.
- A test set for an LLM feature already exists, and nobody can say what each case is there to catch.

Halcyon is a payments API vendor. It has about two hundred staff. Its customers are developers who integrate card payments. They use a date-versioned API and public documentation. Their questions land in a support queue. A small team answers them. The support agreement promises a first response within one hour when money is affected.

The team has put a language model in front of the queue. Every new ticket gets a one-sentence summary and a severity, P1 to P3. The on-call engineer works from those two fields, not from the ticket. That triage assistant is what we evaluate in this series. Later we add a documentation assistant that answers developer questions from the same pages.

On a Thursday afternoon a new ticket arrives. The assistant's summary says a customer wants to cancel pending webhook retries after an outage on their side. Severity P3. First response within one business day. The ticket waits behind the other P3s until Friday.

The ticket itself is a full paragraph. Near the end, the developer mentions something else. A few orders from that afternoon were charged twice. They add that this is probably unrelated. They promise to refund the orders themselves. Under Halcyon's own rules a duplicate charge is P1. One-hour response, however calmly it is reported. Nobody lied. The summary was accurate. The money just was not in it.

Halcyon is fictional. Every page and every ticket in this series is made up. We wrote them for this series, and this post is where we do it. We used to treat this step as housekeeping. We no longer do. An evaluation is a test. A test can only fail on inputs that contain a failure. So building the inputs is where most of the thinking goes. It is also the step most tutorials skip.

There is no metric in this post. That is on purpose. Teams reach for a score before they have anything worth scoring. We wanted the data first, with its traps written down and shown to work. The first metric arrives in the next post.

## The real tickets stay in the ticketing system

Real support tickets are full of things that must not leave the ticketing system. Card fragments. Customer names. Request ids that trace back to accounts. Now and then a live API key that a developer pasted in while debugging.

An evaluation suite has none of the access controls around that system. The engineers who write it read it. The CI runner logs it. The vendor whose model grades it receives it. The auditor asks for it. Sooner or later it ends up in a GitHub repository. So real tickets cannot go in.

Redacting them does not fix this. Scrub lightly and something slips through. Scrub hard and the ids, amounts and pasted details are gone. Those details are what made the ticket a hard case. What is left is safe and useless.

That leaves made-up data. Most teams use it. Almost nobody writes about it. The risk is not that it looks fake. The risk is who writes it. The people who built the system write the test cases. We write cases the way we imagine the system being used. So the system passes, because the data agrees with it.

What we needed to keep was not the look of real tickets. It was the ways real tickets go wrong. A page that is right for one customer and wrong for another. A rule stated once, on a page nobody reads. A customer who buries the important thing at the end. A question the documentation does not answer. A corpus without these can only confirm what already works. A corpus that has them but does not say so will lose them the first time someone trims the suite.

We think about this the way we think about unit tests. A suite of happy-path inputs proves the code runs. The test that earns its place carries the bad input we once saw in production. There is one difference. A unit test has an exact oracle. The function returns 42 or it does not. Here the oracle is a rule in a policy page, applied to prose, by a model. So a trap is not an assertion we can write in advance. It is a case built so that one specific mistake shows up in the output. We do not call it built until we have watched that happen.

Setup first. The notebook reads `GEMINI_API_KEY` from the environment. A full run costs cents, not dollars. The exact bill is printed near the end.

In [1]:
# !uv pip install deepeval==4.2.1 langgraph==1.2.11 langchain-core==1.6.2 langchain-google-genai==4.4.0 google-genai==2.22.0
import os, re, json, time, hashlib, logging, statistics
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version
from typing import TypedDict

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from deepeval.dataset import Golden

logging.getLogger("google_genai.models").setLevel(logging.ERROR)   # silence an SDK advisory about tool calling

if not os.environ.get("GEMINI_API_KEY"):
    raise RuntimeError("Set GEMINI_API_KEY in the environment before running this notebook.")

JUDGE_MODEL = "gemini-3.6-flash"      # the judge every later post uses; this post never calls it
APP_MODEL = "gemini-3.6-flash"        # the model inside the assistant under test
MAX_CONCURRENCY = 4                   # model calls in flight at any moment
PRICE_IN, PRICE_OUT = 0.75 / 1e6, 3.75 / 1e6   # USD per token, gemini-3.6-flash standard tier, 2026 rate
T0 = time.time()
USAGE = []                            # one usage_metadata dict per model call, for the cost ledger


def with_retry(fn, *args, attempts=5, base_delay=2.0):
    """Call fn(*args); on a 429 or 503 wait 2, 4, 8, 16 seconds and try again."""
    for attempt in range(attempts):
        try:
            return fn(*args)
        except Exception as exc:
            transient = any(code in str(exc) for code in ("429", "503", "RESOURCE_EXHAUSTED"))
            if not transient or attempt == attempts - 1:
                raise
            time.sleep(base_delay * 2 ** attempt)


# No temperature, top_p or top_k: Google lists them as deprecated sampling parameters to strip from requests.
llm = ChatGoogleGenerativeAI(model=APP_MODEL, google_api_key=os.environ["GEMINI_API_KEY"], max_retries=0)
for pkg in ("deepeval", "langgraph", "langchain-core", "langchain-google-genai", "google-genai"):
    print(f"{pkg:<24}{version(pkg)}")
print(f"\napp model {APP_MODEL} | judge model {JUDGE_MODEL} | concurrency cap {MAX_CONCURRENCY}")
print(f"temperature the client will send: {llm.temperature}")

deepeval                4.2.1
langgraph               1.2.11
langchain-core          1.6.2
langchain-google-genai  4.4.0
google-genai            2.22.0

app model gemini-3.6-flash | judge model gemini-3.6-flash | concurrency cap 4
temperature the client will send: None


Five pinned versions. Two model constants. One line we care about more than it looks: `temperature the client will send: None`. We set no temperature, top_p or top_k anywhere in this notebook. Google's model guide now lists all three as deprecated. It says to strip them from requests. `langchain-google-genai` 4.4.0 treats `gemini-3.6-flash` as a fixed-sampling model. It sends no temperature for a Gemini 3 model unless one is set. For this model it drops all three with a warning even then. The cell prints what the client will send, so this is on record rather than assumed. Where we need determinism, we get it from instructions. Where the model still varies, we measure the variation instead of trying to switch it off.

The two constants name two roles. `APP_MODEL` is the model inside the thing we are testing. `JUDGE_MODEL` is the model that will grade outputs from the next post onward. A judge is a second model call. It reads an output and scores it against a rubric. Both constants are `gemini-3.6-flash`. Newer Flash models have shipped since. We keep this one for the whole series so that numbers stay comparable across posts. Nothing in this post calls the judge. The constant is here because we re-create this block word for word in every later notebook. `with_retry()` waits and retries on a 429 or 503. `MAX_CONCURRENCY` caps how many calls run at once. `USAGE` collects the token count of every call. That is how the cost at the end is measured rather than estimated.

## Twelve pages, and the three sentences that make them dangerous

Halcyon's public documentation is twelve pages. Each page is a dict with a `slug`, a `title`, a `status` of current or deprecated, an `updated` date and a `body`. Each body is roughly a hundred words. We kept them short on purpose. A retrieval system needs enough text to rank on. A retrieval system is the search that decides which pages the model gets to see. A reader needs little enough text to read in full here.

Three sentences carry the traps. The last sentence of `webhook-signatures-v1`. That page describes HMAC-SHA1 verification. HMAC is a keyed hash. It lets a receiver prove a message came from the sender. The `Idempotency-Key` sentence in `api-conventions`. Idempotency means a request sent twice has the effect of being sent once. And the P1 rule in `support-sla`. The cell after this one checks each of the three mechanically.

In [2]:
DOCS = [
    {"slug": "quickstart", "title": "Quickstart", "status": "current", "updated": "2026-03-01",
     "body": "Create an account, then generate a test API key from the dashboard; test keys start with "
             "sk_test_ and live keys with sk_live_. Install the SDK with pip install halcyon (the current "
             "SDK is 3.x). Create your first charge with POST /v1/charges, passing amount in minor units (an "
             "integer: 1999 means 19.99), a lowercase ISO 4217 currency such as usd or eur, and a source "
             "token from Halcyon.js. Send the Halcyon-Version header with the API version you tested "
             "against so later changes do not reach you until you opt in. Use test card 4242 4242 4242 "
             "4242 for a successful payment and 4000 0000 0000 0002 for a decline."},
    {"slug": "api-conventions", "title": "API conventions", "status": "current", "updated": "2026-03-01",
     "body": "The API is served over HTTPS at https://api.halcyon.dev and is date-versioned: send "
             "Halcyon-Version: 2026-03-01 (or an earlier version) with each request; requests without the "
             "header use the account's default version. Amounts are integers in minor units and currencies "
             "are lowercase ISO 4217 codes. Every POST to a money-moving endpoint (charges, refunds, "
             "payouts) must include an Idempotency-Key header, a client-generated unique string; a UUID v4 "
             "is recommended. Replaying a request with the same key within 24 hours returns the original "
             "response instead of moving money again. Requests without the header are accepted for "
             "backward compatibility, so a retried POST without one creates a second charge. Keys are "
             "scoped to the account and to the endpoint."},
    {"slug": "authentication", "title": "Authentication", "status": "current", "updated": "2026-01-15",
     "body": "Authenticate every request with an Authorization: Bearer header carrying your secret key. "
             "Test-mode keys (sk_test_) never move real money; live-mode keys (sk_live_) do. Restricted "
             "keys can be limited to read-only access or to specific resources. Rotate a key from the "
             "dashboard: the new key is active immediately and the old key keeps working for 24 hours so "
             "deployments can roll over without downtime; revoke it sooner from the same screen if it was "
             "exposed. A missing or invalid key returns HTTP 401 with type authentication_error. Never "
             "send secret keys from browsers or mobile apps; use Halcyon.js publishable keys (pk_) there."},
    {"slug": "errors", "title": "Errors", "status": "current", "updated": "2026-03-01",
     "body": "Errors return a JSON body with type, code, message and request_id. Types map to HTTP "
             "status: invalid_request_error (400), authentication_error (401), card_error (402), "
             "rate_limit_error (429) and api_error (500 to 504). card_error codes include card_declined, "
             "insufficient_funds, expired_card and incorrect_cvc; show the cardholder a generic message "
             "and let them try another card. Requests that return 5xx may have partially completed: retry "
             "them with exponential backoff, and quote the request_id when you contact support. Do not "
             "retry 4xx errors other than 429."},
    {"slug": "rate-limits", "title": "Rate limits", "status": "current", "updated": "2025-11-01",
     "body": "Live mode allows 100 requests per second per account with short bursts to 200; test mode "
             "allows 25 requests per second. Over the limit, requests fail with HTTP 429, type "
             "rate_limit_error, and a Retry-After header giving the number of seconds to wait. Back off "
             "exponentially with jitter rather than retrying in a tight loop, and spread bulk work such as "
             "migrations or invoice runs over time or through a queue. Webhook deliveries to your endpoint "
             "do not count against the limit. Contact support with your expected peak if you need a "
             "higher limit."},
    {"slug": "charges-create", "title": "Create a charge", "status": "current", "updated": "2026-03-01",
     "body": "POST /v1/charges creates a charge. Required parameters: amount (integer, minor units), "
             "currency (lowercase ISO 4217) and source (a token from Halcyon.js or a saved payment method "
             "id starting pm_). Optional: description, metadata (up to 20 key-value pairs), and capture, "
             "which defaults to true; pass capture=false to authorize only, then capture within 7 days "
             "with POST /v1/charges/{id}/capture. The response is a charge object with id (ch_), status "
             "(succeeded, pending or failed), amount, currency, amount_refunded and failure_code. Card "
             "declines return HTTP 402 with a card_error body; a charge that could not be attempted at "
             "all returns 400."},
    {"slug": "refunds", "title": "Refunds", "status": "current", "updated": "2026-02-10",
     "body": "POST /v1/refunds with charge (a ch_ id) refunds a charge in full; pass amount to refund "
             "part of it. A charge can be refunded several times until the refunded total reaches the "
             "original amount; a request beyond the remaining balance returns 400 with code "
             "amount_exceeds_refundable. Refunds of a pending charge are rejected until the charge "
             "succeeds. The refund object has id (re_), status (pending, succeeded or failed) and amount. "
             "Refunds settle to the cardholder in 5 to 10 business days depending on the issuing bank; "
             "status succeeded means Halcyon has released the funds, not that the cardholder has seen "
             "them yet."},
    {"slug": "webhooks-overview", "title": "Webhooks", "status": "current", "updated": "2026-03-01",
     "body": "Halcyon notifies your endpoint about events by POSTing a JSON event object with id (evt_), "
             "type, created and data. Event types include charge.succeeded, charge.failed, "
             "refund.succeeded and dispute.opened. Your endpoint must return a 2xx within 10 seconds; "
             "otherwise the delivery is retried with exponential backoff (1 minute, 5 minutes, 30 minutes, "
             "2 hours, then every 6 hours) for up to 72 hours, after which the event is marked failed and "
             "shown in the dashboard. Delivery is at-least-once: the same event can arrive more than once "
             "and events can arrive out of order, so store the event id and ignore repeats. Always verify "
             "the signature header before trusting an event; see the webhook signatures pages."},
    {"slug": "webhook-signatures-v1", "title": "Webhook signatures (v1)", "status": "deprecated",
     "updated": "2026-03-01",
     "body": "Each webhook request carries an X-Halcyon-Signature header containing a hex-encoded "
             "HMAC-SHA1 of the raw request body, keyed with your endpoint's signing secret (whsec_). To "
             "verify, compute HMAC-SHA1 over the exact bytes you received, before any JSON parsing, and "
             "compare with the header using a constant-time comparison. Reject the event if the values "
             "differ. Do not re-serialize the JSON before hashing, since key ordering and whitespace change "
             "the digest. This scheme was deprecated on 2026-03-01 in favour of webhook-signatures-v2, "
             "which uses HMAC-SHA256 with a timestamped Halcyon-Signature header; it remains in service "
             "for accounts pinned to API versions before 2026-03-01."},
    {"slug": "webhook-signatures-v2", "title": "Webhook signatures (v2)", "status": "current",
     "updated": "2026-03-01",
     "body": "Each webhook request carries a Halcyon-Signature header of the form t=<unix timestamp>,"
             "v2=<hex digest>. The signed payload is the timestamp, a period, and the raw request body. "
             "To verify, compute HMAC-SHA256 of that payload with your endpoint's signing secret (whsec_), "
             "compare with the v2 value using a constant-time comparison, and reject the event if the "
             "timestamp is more than 300 seconds old, which defeats replay of captured requests. During a "
             "signing secret rotation the header carries two v2 values for 24 hours; accept the event if "
             "either matches. This is the default scheme for API version 2026-03-01 and later, and the "
             "only scheme SDK 3.x verifies."},
    {"slug": "changelog-2026-03", "title": "Changelog: API version 2026-03-01", "status": "current",
     "updated": "2026-03-01",
     "body": "API version 2026-03-01. Webhook signatures v2 (HMAC-SHA256, timestamped Halcyon-Signature "
             "header) are now the default; v1 (HMAC-SHA1, X-Halcyon-Signature) is deprecated and is sent "
             "only to accounts pinned to earlier versions. Halcyon SDK 3.0 ships with v2 verification "
             "helpers and no longer verifies v1 headers; SDK 2.x continues to verify v1 only. The "
             "Idempotency-Key replay window is extended from 1 hour to 24 hours, and reusing a key with a "
             "different request body now returns 400 with code idempotency_key_reused instead of silently "
             "returning the earlier response. Request and response shapes for charges and refunds are "
             "unchanged."},
    {"slug": "support-sla", "title": "Support and SLA", "status": "current", "updated": "2026-01-15",
     "body": "Support is available to all accounts through the dashboard and support@halcyon.dev. "
             "Tickets are triaged into three priorities. P1: any report that funds have been affected, "
             "including duplicate or double charges, charges of the wrong amount, and refunds or payouts "
             "that did not arrive, regardless of how the report is worded or whether the ticket is mainly "
             "about something else; first response within 1 hour, 24 hours a day. P2: a live-mode "
             "integration that is failing, such as authentication errors, webhook deliveries or signature "
             "verification failing, or unexpected 4xx or 5xx responses from live endpoints; first response "
             "within 4 business hours. P3: how-to questions, documentation gaps, feature requests and "
             "anything in test mode; first response within 1 business day."},
]
print(f"{len(DOCS)} pages\n")
print(f"{'slug':<24}{'status':<12}{'updated':<12}{'words':>5}")
for d in DOCS:
    print(f"{d['slug']:<24}{d['status']:<12}{d['updated']:<12}{len(d['body'].split()):>5}")

12 pages

slug                    status      updated     words
quickstart              current     2026-03-01    108
api-conventions         current     2026-03-01    114
authentication          current     2026-01-15    100
errors                  current     2026-03-01     78
rate-limits             current     2025-11-01     93
charges-create          current     2026-03-01     95
refunds                 current     2026-02-10    101
webhooks-overview       current     2026-03-01    113
webhook-signatures-v1   deprecated  2026-03-01     98
webhook-signatures-v2   current     2026-03-01    105
changelog-2026-03       current     2026-03-01     94
support-sla             current     2026-01-15    118


Twelve pages, 78 to 118 words each. One is marked deprecated. Reading the traps is not the same as checking them. The next cell searches the pages by string, the way the first stage of a retrieval system would. It prints the sentences the traps live in.

In [3]:
def pages_containing(term):
    return [d["slug"] for d in DOCS if term.lower() in d["body"].lower()]

def page(slug):
    return next(d for d in DOCS if d["slug"] == slug)

def vocabulary(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))

print("Pages that mention Idempotency-Key:   ", pages_containing("Idempotency-Key"))
print("Pages that mention POST /v1/charges:  ", pages_containing("POST /v1/charges"))
print("Pages that mention HMAC:              ", pages_containing("HMAC"))

v1, v2 = page("webhook-signatures-v1"), page("webhook-signatures-v2")
shared = len(vocabulary(v1["body"]) & vocabulary(v2["body"]))
union = len(vocabulary(v1["body"]) | vocabulary(v2["body"]))
print(f"\nDistinct words the two signature pages share: {shared} of {union} ({shared / union:.0%})")
print(f"Position of the word 'deprecated' in the v1 page: "
      f"{v1['body'].lower().index('deprecated') / len(v1['body']):.0%} of the way through")
print("Last sentence of webhook-signatures-v1:")
print("  " + v1["body"].split(". ")[-1])

print("\nsupport-sla, the P1 rule:")
print("  " + re.search(r"P1: [^.]*\.", page("support-sla")["body"]).group(0))

Pages that mention Idempotency-Key:    ['api-conventions', 'changelog-2026-03']
Pages that mention POST /v1/charges:   ['quickstart', 'charges-create']
Pages that mention HMAC:               ['webhook-signatures-v1', 'webhook-signatures-v2', 'changelog-2026-03']

Distinct words the two signature pages share: 46 of 119 (39%)
Position of the word 'deprecated' in the v1 page: 70% of the way through
Last sentence of webhook-signatures-v1:
  This scheme was deprecated on 2026-03-01 in favour of webhook-signatures-v2, which uses HMAC-SHA256 with a timestamped Halcyon-Signature header; it remains in service for accounts pinned to API versions before 2026-03-01.

support-sla, the P1 rule:
  P1: any report that funds have been affected, including duplicate or double charges, charges of the wrong amount, and refunds or payouts that did not arrive, regardless of how the report is worded or whether the ticket is mainly about something else; first response within 1 hour, 24 hours a day.


`Idempotency-Key` appears on two pages: `api-conventions` and `changelog-2026-03`. `POST /v1/charges` appears on two pages: `quickstart` and `charges-create`. The lists do not overlap. A developer searching for how to create a charge lands on two pages. Neither page says the header is required. The header is what prevents a duplicate charge on retry. The conventions page adds that requests without it are accepted. So an assistant that answers "how do I create a charge" from the charge page alone gives an accurate answer. It is well sourced. It will double-charge a customer the first time the client retries a timeout. When we get to retrieval later in the series, this is the failure we look for on the answer side. The search fetched a correct page. The answer was faithful to it. The page was incomplete.

The two signature pages share 46 of their 119 distinct words. That is 39 percent. The word "deprecated" appears 70 percent of the way through the v1 page, in its last sentence. Both pages are published, because customers on older API versions still receive v1 headers. A system that ranks pages by similarity to a question has no reason to prefer one over the other. A question about verifying webhook signatures matches both. The retrieval post shows what happens when it picks the wrong one. A developer in 2026 is told to verify with SHA1. Every sentence of that answer is backed by a published page.

The P1 rule is one blunt sentence. Funds affected means P1. It does not matter how the report is worded. It does not matter if the ticket is mainly about something else. We wrote that clause for a reason. People who have just been double-charged tend to mention it in passing, while asking about something else. The rule is only as good as the text it is applied to. That brings us to the tickets.

## Twelve tickets, four of them P1 no matter how they sound

The tickets are inbound developer support. Each has an `id`, a `submitted` date, a `subject` and a `body` in the developer's voice. Four of them mention duplicate charges: T-1003, T-1006, T-1009 and T-1011. In all four, the money is the second problem in a ticket about something else. A webhook retry storm. A failed SDK upgrade. A batch import that times out. Intermittent 502s. The developer leads with the integration problem. The double charge comes late. They play it down with "probably unrelated" or "somewhat related". They usually offer to sort it out themselves.

We spent more time on these four than on the other eight put together. The placement is the whole trap. If the duplicate charge were the subject line, no summary would ever drop it. There would be nothing to catch. Real developers write about the problem that blocks them. The money is Halcyon's problem, and they say so. So we wrote the four so that a faithful one-sentence summary has to choose what to keep. Below, we check what it chooses.

The other eight are uneven on purpose. Two ask about things no page covers: disputes in test mode (T-1008) and wallet payments (T-1012). They give a documentation assistant room to fail by inventing an answer. T-1010 uses "duplicate" and "double" about webhook deliveries. No money is involved. It gives a keyword rule something to trip on. T-1001 has two correct answers, depending on the account's API version. The rest are plain how-to questions. We need those too. A failing suite must be able to tell "the trap fired" from "everything is broken".

In [4]:
TICKETS = [
    {"id": "T-1001", "submitted": "2026-08-03",
     "subject": "Which signature header am I supposed to verify?",
     "body": "We're integrating webhooks and every event we receive has an X-Halcyon-Signature header but "
             "no Halcyon-Signature header, so the verify() helper in SDK 3.1 throws 'no v2 signature "
             "present'. The webhook overview says to verify signatures and links to two pages. Our account "
             "was created in 2025 and I think we're pinned to 2025-11-01. Is v1 still supported, do we need "
             "to move to v2, and how do we change the API version without breaking our existing charge "
             "flow? This is blocking our launch on the live account."},
    {"id": "T-1002", "submitted": "2026-08-04",
     "subject": "429s in test mode when our integration suite runs",
     "body": "Our CI runs about 40 requests per second against test mode at peak and we get bursts of "
             "rate_limit_error with Retry-After: 1. Live mode is fine. Is the test-mode limit documented "
             "anywhere, and can it be raised for CI? We can add a throttle but it makes the suite roughly "
             "three times slower."},
    {"id": "T-1003", "submitted": "2026-08-05",
     "subject": "Retry storm after our webhook endpoint returned 500s",
     "body": "On Thursday a bad deploy made our webhook endpoint return 500 for about forty minutes. Since "
             "then we've received roughly 4,000 redeliveries, some for events from days ago, and they are "
             "still trickling in. Is there a way to cancel pending retries for an endpoint, or do we just "
             "absorb them until the 72 hours are up? Probably unrelated, but while going through the logs "
             "we found a few orders from that afternoon that were charged twice, two charge ids each; we "
             "assume our order service retried the charge call on timeout and we'll refund them on our "
             "side. Mainly we need the retries to stop."},
    {"id": "T-1004", "submitted": "2026-08-06",
     "subject": "Getting 400 idempotency_key_reused on retries",
     "body": "Since we moved to API version 2026-03-01 last week, our retry path fails with 400 "
             "idempotency_key_reused. We generate one Idempotency-Key per order and reuse it if the first "
             "attempt fails, which used to work. Looking closer, our retry adds a metadata.attempt field "
             "to the body. Is the body compared exactly? What is the recommended pattern for retries where "
             "the body legitimately changes? This is live traffic; failed retries currently land in a "
             "dead-letter queue and someone replays them by hand."},
    {"id": "T-1005", "submitted": "2026-08-07",
     "subject": "Partial refund returns amount_exceeds_refundable",
     "body": "A customer paid 120.00 EUR and we refunded 80.00 EUR last week. Refunding the remaining 40.00 "
             "EUR today returns 400 amount_exceeds_refundable. It turns out our cancellation flow had "
             "already issued a separate 40.00 EUR refund that nobody noticed, so the error is correct. "
             "What we actually need: is there a field on the charge that tells us the remaining refundable "
             "balance, so we can check before calling refunds? And is there any per-month cap on refunds?"},
    {"id": "T-1006", "submitted": "2026-08-10",
     "subject": "SDK 3.0 upgrade broke webhook verification, rolled back",
     "body": "We upgraded from SDK 2.9 to 3.0 on Monday. From the first deploy every webhook failed "
             "verification with 'no v2 signature present' and our fulfilment queue stalled. We rolled back "
             "to 2.9 after about 25 minutes and verification works again. Our account is on API version "
             "2025-11-01. What exactly changed in 3.0, and what is the migration order: SDK first or API "
             "version first? One side effect: while the queue was stalled a colleague re-ran the stuck jobs "
             "by hand and two customers were charged a second time; we've refunded one and are chasing the "
             "other. The verification question is the one we need answered before we try again."},
    {"id": "T-1007", "submitted": "2026-08-11",
     "subject": "Rotated our live key but the old one still works",
     "body": "We rotated our live secret key from the dashboard this morning as part of our quarterly "
             "rotation. Two hours later, requests using the old key still succeed. Is rotation delayed, or "
             "did it not take? We expected the old key to stop working straight away. Also, is there an "
             "audit log showing which key made which request?"},
    {"id": "T-1008", "submitted": "2026-08-12",
     "subject": "How do we simulate a dispute in test mode?",
     "body": "We're building our dispute.opened handler and can't find a way to trigger a dispute against a "
             "test-mode charge. Is there a test card number or a dashboard button that opens a dispute? "
             "Also, what fields does the dispute object carry? Not urgent, we're a few weeks from launch."},
    {"id": "T-1009", "submitted": "2026-08-13",
     "subject": "Batch import script times out around row 200",
     "body": "We're migrating 1,800 customers from our old provider. The script imports each saved payment "
             "method and immediately creates the first invoice charge. It runs fine for about 200 rows and "
             "then requests start timing out at 30 seconds; the script retries each failed row up to three "
             "times and eventually finishes, but the whole run takes hours. Is there a batch endpoint, or "
             "a recommended concurrency for imports? Somewhat related: our finance team says a dozen or so "
             "of the imported customers have two invoice charges instead of one, which we think is the "
             "retry path. We can clean that up ourselves once the import is stable."},
    {"id": "T-1010", "submitted": "2026-08-14",
     "subject": "Duplicate webhook deliveries and events arriving out of order",
     "body": "We're seeing the same evt_ id delivered two or three times, sometimes minutes apart, and "
             "occasionally a refund.succeeded arrives before the charge.succeeded for the same order. No "
             "money problem that we can see, every charge is correct in the dashboard, but our handler "
             "assumed one delivery per event and now double-writes some rows. Is this expected behaviour? "
             "Should we be deduplicating on our side and, if so, on which field?"},
    {"id": "T-1011", "submitted": "2026-08-17",
     "subject": "Intermittent 502s from POST /v1/charges since Tuesday",
     "body": "Roughly one in fifty POST /v1/charges calls returns a 502 with an HTML body rather than your "
             "JSON error format, all from the eu-west region. Our client retries 5xx with backoff as your "
             "errors page recommends and the retry almost always succeeds, so checkout is mostly working. "
             "Request ids for six failures are attached. Two customers have emailed our support saying "
             "their statement shows the same order twice, which I assume is the retry; we'll sort that out "
             "with them. The 502s are what we need you to look at."},
    {"id": "T-1012", "submitted": "2026-08-18",
     "subject": "Do you support Apple Pay and Google Pay?",
     "body": "Our product team wants wallet payments at checkout. I can't find Apple Pay or Google Pay "
             "anywhere in the docs. Is it supported through Halcyon.js, on a roadmap, or would we need a "
             "second provider for wallets? Also, is there a fee difference for wallet payments?"},
]

def corpus_fingerprint(docs, tickets):
    return hashlib.sha256(json.dumps([docs, tickets], sort_keys=True).encode()).hexdigest()[:12]
print(f"{len(TICKETS)} tickets\n")
print(f"{'id':<8}{'submitted':<12}{'words':>5}  subject")
for t in TICKETS:
    print(f"{t['id']:<8}{t['submitted']:<12}{len(t['body'].split()):>5}  {t['subject']}")
print(f"\ncorpus fingerprint over DOCS and TICKETS: {corpus_fingerprint(DOCS, TICKETS)}")

12 tickets

id      submitted   words  subject
T-1001  2026-08-03     87  Which signature header am I supposed to verify?
T-1002  2026-08-04     53  429s in test mode when our integration suite runs
T-1003  2026-08-05    106  Retry storm after our webhook endpoint returned 500s
T-1004  2026-08-06     80  Getting 400 idempotency_key_reused on retries
T-1005  2026-08-07     76  Partial refund returns amount_exceeds_refundable
T-1006  2026-08-10    109  SDK 3.0 upgrade broke webhook verification, rolled back
T-1007  2026-08-11     57  Rotated our live key but the old one still works
T-1008  2026-08-12     48  How do we simulate a dispute in test mode?
T-1009  2026-08-13    107  Batch import script times out around row 200
T-1010  2026-08-14     70  Duplicate webhook deliveries and events arriving out of order
T-1011  2026-08-17     90  Intermittent 502s from POST /v1/charges since Tuesday
T-1012  2026-08-18     46  Do you support Apple Pay and Google Pay?

corpus fingerprint over DOCS and

Twelve tickets of 46 to 109 words. At the bottom is the corpus fingerprint: `e4485ba0f187`. We hash `DOCS` and `TICKETS` together and keep the first twelve characters. Every later notebook asserts that value before it does anything else. That way, when we compare numbers across posts, we know they came from the same data. It works like a lockfile. A lockfile pins other people's code. This pins our own data. A matching hash only says nothing moved. It does not say the data is right.

Before spending a model call, we can check one design claim mechanically. Where in each ticket does the money first appear?

In [5]:
MONEY = re.compile(r"twice|double|duplicat|second (charge|time)|two (charge|invoice)|charged again|"
                   r"extra charge|multiple charge|overcharg|billed (again|twice)", re.I)
MONEY_TICKET_IDS = ("T-1003", "T-1006", "T-1009", "T-1011")   # written with the money buried on purpose

def pattern_position(text):
    """Where the first match of MONEY falls, as a fraction of the ticket's words, or None."""
    match = MONEY.search(text)
    if match is None:
        return None
    return len(text[:match.start()].split()) / len(text.split())

print(f"{'ticket':<8}{'words':>5}  {'pattern match':<20}subject")
for t in TICKETS:
    pos = pattern_position(t["body"])
    where = "none" if pos is None else f"at {pos:.0%} of the body"
    print(f"{t['id']:<8}{len(t['body'].split()):>5}  {where:<20}{t['subject']}")

ticket  words  pattern match       subject
T-1001     87  none                Which signature header am I supposed to verify?
T-1002     53  none                429s in test mode when our integration suite runs
T-1003    106  at 72% of the body  Retry storm after our webhook endpoint returned 500s
T-1004     80  none                Getting 400 idempotency_key_reused on retries
T-1005     76  none                Partial refund returns amount_exceeds_refundable
T-1006    109  at 79% of the body  SDK 3.0 upgrade broke webhook verification, rolled back
T-1007     57  none                Rotated our live key but the old one still works
T-1008     48  none                How do we simulate a dispute in test mode?
T-1009    107  at 78% of the body  Batch import script times out around row 200
T-1010     70  at 71% of the body  Duplicate webhook deliveries and events arriving out of order
T-1011     90  at 74% of the body  Intermittent 502s from POST /v1/charges since Tuesday
T-1012     46  no

`pattern_position()` reports where `MONEY` first matches in each body. `MONEY` is a pattern for phrases like "twice", "double" and "duplicate". In all four money tickets, the first match is between 72 and 79 percent of the way through. That is after the developer has finished asking their own question. T-1010 also matches, at 71 percent. Its body says the customer's handler "double-writes" rows. That false positive is one of the traps. It is also why we do not evaluate anything in this series with keyword rules. The pattern is good enough to check whether a summary kept a phrase. That is all we use it for below. It is not good enough to decide whether a ticket is about money.

So the money is buried by construction. Whether a summarizer drops it is a claim about the model. We wanted to see that before building anything on it. The next cell asks gemini-3.6-flash for a one-sentence summary of each of the four tickets. At most 25 words. Twice each. It uses a bare model call with no graph around it. A single call with nothing before or after it does not need one. The prompt is the same one the assistant uses in the next section. It says nothing about the SLA or about money. That is the point. This is the summarizer the way a team would first write it.

In [6]:
SUMMARIZE = ("You write one-line summaries for Halcyon's support triage queue. Summarize the ticket below "
             "in one sentence of at most 25 words, in plain English.\n\nTicket:\n{ticket}")

def ticket_text(ticket):
    return f"Subject: {ticket['subject']}\n\n{ticket['body']}"
PROBE_REPEATS = 2
probe = []
for t in (t for t in TICKETS if t["id"] in MONEY_TICKET_IDS):
    for run in range(1, PROBE_REPEATS + 1):
        reply = with_retry(llm.invoke, [HumanMessage(SUMMARIZE.format(ticket=ticket_text(t)))])
        USAGE.append(reply.usage_metadata or {})
        probe.append((t["id"], run, reply.text.strip()))

kept = 0
for tid, run, summary in probe:
    survived = MONEY.search(summary) is not None
    kept += survived
    print(f"{tid} run {run}  money {'KEPT' if survived else 'LOST'}  {summary}")
print(f"\nThe money survived in {kept} of {len(probe)} one-sentence summaries.")

T-1003 run 1  money LOST  The customer is asking to cancel pending webhook retries for their endpoint after a temporary outage caused a high-volume retry storm.
T-1003 run 2  money LOST  The customer is asking to cancel pending webhook retries after a temporary endpoint outage caused a retry storm.
T-1006 run 1  money LOST  Upgrading to SDK 3.0 broke webhook verification, so the customer asks what changed and the correct SDK versus API version migration order.
T-1006 run 2  money LOST  Upgrading to SDK 3.0 broke webhook verification, and the customer asks what changed and the correct SDK versus API version migration order.
T-1009 run 1  money KEPT  Customer's migration script times out around row 200, causing duplicate charges, and asks for guidance on batch endpoints or optimal concurrency.
T-1009 run 2  money KEPT  Customer's import script times out around row 200, causing duplicate charges, and they are requesting batch endpoints or concurrency guidance.
T-1011 run 1  money KEPT  In

Three of eight summaries kept the money. Every one of the eight is fluent and accurate. Each is a fair one-line account of what the developer asked for. T-1003's two summaries are about cancelling retries. T-1006's are about the SDK and the migration order. T-1009 kept the duplicate charges both times. T-1011 kept them once and dropped them once. That is the model's run-to-run variation. It lands where we would expect, on the sentence that is hardest to fit into 25 words.

What decides survival is not subtle. In T-1009 the duplicate charges are caused by the very retries the developer asks about. They are part of the main story, so the summary keeps them. In T-1003 and T-1006 the developer called the money "probably unrelated" and "one side effect". They promised to handle it and moved on. The summarizer takes the developer's word for what matters. A summary inherits its author's priorities. The SLA exists because the author's priorities are wrong about money.

That is the trap confirmed. Now we build the assistant it is aimed at.

## The assistant under test, and what it forgets

The system under test is the software the evaluations are about. Here it is Halcyon's Ticket Triage Assistant. A ticket goes in. A one-sentence summary and a severity come out. We built it with LangGraph. The graph has a typed state, a `TypedDict` with a `question`, an optional list of `context` pages and an `answer`. It has one node, `respond()`. We expose it through `answer(question, context=None)`. When we pass pages as context, they go into the prompt ahead of the question. That is how the retrieval post later feeds documentation to the same assistant.

One node is more ceremony than a single model call needs. We know. We kept the graph because this object is what every later post evaluates. Later posts also record its calls and use it as a release check. A graph gives it a state schema and one place every model call passes through. A cost ledger hooks in there now. Call recording hooks in there later. `triage()` is two calls through `answer()`. First it summarizes the ticket with the prompt from the previous section. Then it classifies the summary against the `support-sla` page, pulled straight out of `DOCS`. The answer must be exactly P1, P2 or P3.

The classifier sees only the summary. We did not do that to make the demo work. It is how a queue works. The summary is what the on-call engineer reads. It is what the dashboard column shows. It is what a downstream routing rule runs on. Whatever it drops is gone for everything after it.

In [7]:
class AppState(TypedDict):
    question: str
    context: list[str] | None
    answer: str

SYSTEM = ("You are Halcyon's developer support assistant. Halcyon is a payments API vendor. When "
          "documentation pages are supplied, answer from them alone and say so if they do not cover the "
          "question. Be concise and specific.")

def respond(state: AppState) -> dict:
    prompt = state["question"]
    if state.get("context"):
        prompt = "Documentation pages:\n\n" + "\n\n".join(state["context"]) + f"\n\nQuestion: {prompt}"
    reply = with_retry(llm.invoke, [SystemMessage(SYSTEM), HumanMessage(prompt)])
    USAGE.append(reply.usage_metadata or {})
    return {"answer": reply.text.strip()}

builder = StateGraph(AppState)
builder.add_node("respond", respond)
builder.add_edge(START, "respond")
builder.add_edge("respond", END)
APP = builder.compile()

def answer(question, context=None):
    return APP.invoke({"question": question, "context": context})["answer"]

SLA = next(d for d in DOCS if d["slug"] == "support-sla")["body"]
CLASSIFY = ("Assign a support priority to the ticket summary below using Halcyon's SLA. Reply with exactly "
            "one of P1, P2 or P3 and nothing else.\n\nSLA:\n{sla}\n\nTicket summary: {summary}")

def triage(ticket):
    """Ticket in, one-sentence summary plus severity out. The severity step sees only the summary."""
    summary = answer(SUMMARIZE.format(ticket=ticket_text(ticket)))
    verdict = re.search(r"P[123]", answer(CLASSIFY.format(sla=SLA, summary=summary)))
    return {"id": ticket["id"], "summary": summary, "severity": verdict.group(0) if verdict else "unparsed"}
print("graph nodes:", [n for n in APP.get_graph().nodes if not n.startswith("__")])
print("entry points: answer(question, context=None) and triage(ticket)")

graph nodes: ['respond']
entry points: answer(question, context=None) and triage(ticket)


One node, two entry points. `answer()` is what we evaluate later with documentation in context. `triage()` is what the next post evaluates. Both pass through `respond()`, where `USAGE.append()` records every call. Before running the whole set, here is one ticket in full. The ticket, the summary, the severity.

In [8]:
sample = next(t for t in TICKETS if t["id"] == "T-1003")
result = triage(sample)

print(ticket_text(sample))
print("-" * 78)
print("summary : ", result["summary"])
print("severity: ", result["severity"])
print("money in summary:", "yes" if MONEY.search(result["summary"]) else "no")

Subject: Retry storm after our webhook endpoint returned 500s

On Thursday a bad deploy made our webhook endpoint return 500 for about forty minutes. Since then we've received roughly 4,000 redeliveries, some for events from days ago, and they are still trickling in. Is there a way to cancel pending retries for an endpoint, or do we just absorb them until the 72 hours are up? Probably unrelated, but while going through the logs we found a few orders from that afternoon that were charged twice, two charge ids each; we assume our order service retried the charge call on timeout and we'll refund them on our side. Mainly we need the retries to stop.
------------------------------------------------------------------------------
summary :  The customer is asking how to cancel pending webhook retries following an endpoint outage that caused a retry storm.
severity:  P3
money in summary: no


The summary is a correct sentence about the ticket: "The customer is asking how to cancel pending webhook retries following an endpoint outage that caused a retry storm." The severity is P3. First response within one business day. The classifier had the SLA page in its prompt. That page says "regardless of how the report is worded or whether the ticket is mainly about something else". The classifier applied it. The rule cannot fire on text that is not there.

This is the Thursday afternoon from the opening, reproduced on demand. A fluent, faithful step upstream turned a P1 into a P3. Nothing in the output looks wrong. Anyone working this queue sees a routine webhook question. The customer who was double-charged hears from Halcyon on Friday.

## The golden set says why each case exists

To repeat this across twelve tickets, we need a record for each one. What is the right answer? Why is the ticket in the set at all? That record is called a golden. A golden holds an input, the expected output, and whatever else a test needs to decide pass or fail. It is kept apart from the system's actual answer. That way the same golden serves every version of the system. DeepEval's `Golden` object has fields for all of it. The next cell fills them.

In [9]:
# Per ticket: expected severity, trap label, pages that answer it, reference summary, and why it exists.
GOLDEN_SPEC = {
    "T-1001": ("P2", "two-truths",
               ["webhook-signatures-v1", "webhook-signatures-v2", "changelog-2026-03", "api-conventions"],
               "Live webhooks carry only the v1 X-Halcyon-Signature header, so SDK 3.1 verification fails; "
               "asks whether to move the account to API version 2026-03-01.",
               "Both signature pages are published and both are correct for someone; the right answer "
               "depends on the account's API version."),
    "T-1002": ("P3", None, ["rate-limits"],
               "CI hits test-mode rate limits at 40 requests per second; asks whether the limit is "
               "documented or can be raised.",
               "A plain how-to with one page that answers it fully; the control case."),
    "T-1003": ("P1", "money-buried", ["webhooks-overview", "api-conventions"],
               "Webhook retry storm after endpoint 500s; wants pending retries cancelled and reports "
               "several orders charged twice during the incident.",
               "The ticket is about retries; the double charge is an aside at the end. P1 under the SLA."),
    "T-1004": ("P2", None, ["api-conventions", "changelog-2026-03", "errors"],
               "Retries reusing an Idempotency-Key fail with 400 idempotency_key_reused on live traffic "
               "because the retry body differs; asks for the correct retry pattern.",
               "Needs two pages combined: the conventions rule and the changelog entry that changed it."),
    "T-1005": ("P3", None, ["refunds", "charges-create"],
               "Second partial refund rejected with amount_exceeds_refundable; asks how to read the "
               "remaining refundable balance before refunding.",
               "Mentions refunds and amounts without any funds being affected; a P3 that sounds like money."),
    "T-1006": ("P1", "money-buried", ["changelog-2026-03", "webhook-signatures-v2", "webhook-signatures-v1"],
               "SDK 3.0 upgrade made webhook verification fail and was rolled back; two customers were "
               "charged twice during the stall; asks the migration order.",
               "Leads with a failed upgrade; the double charge is a side effect the developer plays down."),
    "T-1007": ("P3", None, ["authentication"],
               "After a scheduled key rotation the old live key still works two hours later; asks whether "
               "that is expected and for how long.",
               "Sounds like a security incident and is documented behaviour; tests calm reading of docs."),
    "T-1008": ("P3", "no-coverage", [],
               "Cannot trigger a dispute against a test-mode charge; asks how to simulate one and what the "
               "dispute object contains.",
               "No page covers this. The correct answer is to say so, not to invent a test card."),
    "T-1009": ("P1", "money-buried", ["rate-limits", "api-conventions", "charges-create"],
               "Bulk import of 1,800 customers times out after about 200 rows; retries have created "
               "duplicate invoice charges for a dozen customers.",
               "Leads with timeouts and batch advice; the duplicate charges are 'somewhat related'."),
    "T-1010": ("P3", "near-miss", ["webhooks-overview"],
               "Same webhook event delivered several times and out of order; no money affected; asks "
               "whether that is expected and how to deduplicate.",
               "Uses 'duplicate' and 'double' about deliveries, not charges. A keyword rule fires; the "
               "SLA does not."),
    "T-1011": ("P1", "money-buried", ["errors", "charges-create", "api-conventions"],
               "About one in fifty live POST /v1/charges calls returns 502 from eu-west; retries have "
               "charged two customers twice.",
               "Leads with 502s and follows the errors page's own retry advice; the retry is what "
               "double-charged customers."),
    "T-1012": ("P3", "no-coverage", [],
               "Asks whether Apple Pay and Google Pay are supported through Halcyon.js or planned.",
               "No page covers wallets. Tests whether the assistant admits a documentation gap."),
}

GOLDENS = []
for t in TICKETS:
    severity, trap, pages, summary, reason = GOLDEN_SPEC[t["id"]]
    GOLDENS.append(Golden(name=t["id"], input=ticket_text(t), expected_output=summary,
                          additional_metadata={"expected_severity": severity, "trap": trap,
                                               "pages": pages, "reason": reason}))
print(f"{len(GOLDENS)} goldens")
print("expected severities:", dict(sorted(Counter(g.additional_metadata["expected_severity"] for g in GOLDENS).items())))
print("traps:", dict(Counter(str(g.additional_metadata["trap"]) for g in GOLDENS)))
g = GOLDENS[2]
print(f"\n{g.name}: expected {g.additional_metadata['expected_severity']}, trap {g.additional_metadata['trap']}")
print("reference summary:", g.expected_output)
print("why it exists:    ", g.additional_metadata["reason"])

12 goldens
expected severities: {'P1': 4, 'P2': 2, 'P3': 6}
traps: {'two-truths': 1, 'None': 4, 'money-buried': 4, 'no-coverage': 2, 'near-miss': 1}

T-1003: expected P1, trap money-buried
reference summary: Webhook retry storm after endpoint 500s; wants pending retries cancelled and reports several orders charged twice during the incident.
why it exists:     The ticket is about retries; the double charge is an aside at the end. P1 under the SLA.


Twelve goldens: four P1, two P2, six P3. Each carries three things beyond the input and expected output. `expected_severity` is checked with a string comparison and no judge. A question with a fixed answer should never be handed to a model. `pages` lists the documentation that answers the ticket. The coverage check below runs on it. `reason` is one sentence on why the case exists. `expected_output` is a reference summary we wrote by hand. It keeps the money. The next post uses it for metrics that compare an output against a reference.

Eight of twelve goldens carry a named trap. `money-buried` on four. `no-coverage` on two. `near-miss` on one. `two-truths` on one. The remaining four are the controls.

The `reason` field is the one teams skip. We treat it as the cheapest insurance in the file. From its subject line, T-1010 looks like a repeat of the webhooks theme that T-1003 already covers. An engineer trimming the suite a year from now would cut it. Its reason says it is the near-miss. It says "duplicate" with no money at stake. It is there so that a keyword rule fires where the SLA does not. A golden with its reason written down survives a pruning pass. One whose reason lives in the author's head does not.

In [10]:
with ThreadPoolExecutor(MAX_CONCURRENCY) as pool:
    results = list(pool.map(triage, TICKETS))

by_id = {g.name: g for g in GOLDENS}
matched, money_kept = 0, 0
for r in results:
    meta = by_id[r["id"]].additional_metadata
    ok = r["severity"] == meta["expected_severity"]
    matched += ok
    note = ""
    if meta["trap"] == "money-buried":
        survived = MONEY.search(r["summary"]) is not None
        money_kept += survived
        note = "  [money KEPT]" if survived else "  [money LOST]"
    print(f"{r['id']}  expected {meta['expected_severity']}  got {r['severity']}  "
          f"{'ok  ' if ok else 'MISS'}{note}")
    print(f"        {r['summary']}")

print(f"\nseverity matched the golden on {matched} of {len(results)} tickets")
print(f"money survived the summary on {money_kept} of {len(MONEY_TICKET_IDS)} money-buried tickets")

T-1001  expected P2  got P3  MISS
        Webhook verification fails due to signature header mismatch, blocking launch; customer asks how to safely update their API version.
T-1002  expected P3  got P3  ok  
        The customer is hitting test-mode rate limits during CI runs and asks if the limit can be increased.
T-1003  expected P1  got P3  MISS  [money LOST]
        The customer is asking how to cancel pending webhook retries after an endpoint outage triggered a retry storm.
T-1004  expected P2  got P3  MISS
        Customer gets 400 idempotency_key_reused errors when retrying requests with modified request bodies on API version 2026-03-01.
T-1005  expected P3  got P3  ok  
        Merchant asks if a charge field indicates remaining refundable balances and whether there is a per-month cap on refunds.
T-1006  expected P1  got P2  MISS  [money LOST]
        After an SDK 3.0 upgrade broke webhook verification, the customer asks what changed and the correct SDK and API version migratio

Seven of twelve severities matched the golden. The five misses come in three kinds. T-1003 and T-1006 lost the money in the summary. They came out P3 and P2. That is the trap, twice. T-1011 lost the money this time and came out P2, for the 502s. That is the SLA applied correctly to a summary missing its most important fact. T-1001 and T-1004 came out P3 against an expected P2. No money was involved there. The classifier simply read the SLA's P2 wording differently from us. Only T-1009 kept the money and got P1. As in the single instance, the severity on the four money tickets tracked the summary exactly. Money present, P1. Money absent, the severity of whatever the summary did keep.

Every number in this cell moves between runs. Both steps are model calls. We ran these same cells once before, while drafting. That run produced the identical per-ticket money pattern. It produced a different set of misses at the P2 and P3 boundary. The direction holds. Money tickets that lose the money never come out P1. The classifier wavers between P2 and P3, and only on tickets that are not about money. So the next cell stops looking at one run. It pools every summary this notebook has produced.

In [11]:
samples = [(tid, s) for tid, _, s in probe] + [(result["id"], result["summary"])]
samples += [(r["id"], r["summary"]) for r in results]

print("money-buried tickets: did the money survive, across every summary this notebook produced")
for tid in MONEY_TICKET_IDS:
    outcomes = [MONEY.search(s) is not None for t, s in samples if t == tid]
    print(f"  {tid}  kept {sum(outcomes)} of {len(outcomes)}   {' '.join('kept' if o else 'lost' for o in outcomes)}")
overall = [MONEY.search(s) is not None for t, s in samples if t in MONEY_TICKET_IDS]
print(f"\noverall: the money survived in {sum(overall)} of {len(overall)} summaries")

money-buried tickets: did the money survive, across every summary this notebook produced
  T-1003  kept 0 of 4   lost lost lost lost
  T-1006  kept 0 of 3   lost lost lost
  T-1009  kept 3 of 3   kept kept kept
  T-1011  kept 1 of 3   kept lost lost

overall: the money survived in 4 of 13 summaries


Thirteen summaries of the four money tickets, from the probe, the single instance and the full run. Four kept the money. T-1003: zero of four. T-1006: zero of three. T-1009: three of three. T-1011: one of three.

This is what we mean when we say a trap fires. Not one dropped summary, which could be luck. A pattern that repeats per ticket, in the same direction. T-1003 and T-1006 fire reliably. T-1011 is roughly a coin flip. T-1009 does not fire at all. We are keeping T-1009 in the set, still labelled `money-buried`. A trap that fires every time tells us nothing about where the boundary is. The next post needs a ticket the summarizer gets right as much as the ones it gets wrong.

## What this corpus can catch, and what it cannot

Our coverage check asks four questions. Is every documentation area represented? Do the tickets exercise the whole documentation set? Is there an unhappy path, where the right answer is not an answer? What can this corpus not detect? The first three we compute from the `pages` field on each golden. The fourth we have to answer ourselves. Code can show part of it.

In [12]:
page_hits = {d["slug"]: [] for d in DOCS}
uncovered = []
for g in GOLDENS:
    pages = g.additional_metadata["pages"]
    if not pages:
        uncovered.append(g.name)
    for slug in pages:
        page_hits[slug].append(g.name)

print("Tickets that exercise each documentation page")
for slug, ids in page_hits.items():
    print(f"  {slug:<24}{len(ids):>2}  {' '.join(ids)}")
print("\nPages no ticket exercises:", [slug for slug, ids in page_hits.items() if not ids])
print("Tickets no page answers:  ", uncovered)
print("Tickets with a planted trap:", sum(g.additional_metadata["trap"] is not None for g in GOLDENS),
      "of", len(GOLDENS))

Tickets that exercise each documentation page
  quickstart               0  
  api-conventions          5  T-1001 T-1003 T-1004 T-1009 T-1011
  authentication           1  T-1007
  errors                   2  T-1004 T-1011
  rate-limits              2  T-1002 T-1009
  charges-create           3  T-1005 T-1009 T-1011
  refunds                  1  T-1005
  webhooks-overview        2  T-1003 T-1010
  webhook-signatures-v1    2  T-1001 T-1006
  webhook-signatures-v2    2  T-1001 T-1006
  changelog-2026-03        3  T-1001 T-1004 T-1006
  support-sla              0  

Pages no ticket exercises: ['quickstart', 'support-sla']
Tickets no page answers:   ['T-1008', 'T-1012']
Tickets with a planted trap: 8 of 12


Two pages have no ticket. `quickstart`, because developers who need it have not filed a ticket yet. `support-sla`, because it is Halcyon's internal rule, not something developers ask about. It reaches the assistant through the classifier prompt instead. `api-conventions` is exercised by five tickets. That is about right. The idempotency rule is the root cause under three of the money tickets. Two tickets have no page: T-1008 and T-1012. They are the unhappy path for a documentation assistant. The correct answer is to say the documentation does not cover this. An assistant that would rather be helpful than right invents a test card number. Eight of twelve tickets carry a planted trap.

Then the fourth question. Twelve pages cannot show the failures that appear only at scale. At scale, a retrieval system must pick the right page out of thousands. It gets the ranking slightly wrong, rather than confusing two near-twins. Our v1 and v2 trap is a near-tie between two pages, not a needle in a haystack. And twelve tickets written in one sitting share a voice. Real inbound support text does not. That second limit we can at least measure.

In [13]:
def sentences(text):
    return [s for s in re.split(r"(?<=[.?!])\s+", text) if s]

mean_lengths = [statistics.mean(len(s.split()) for s in sentences(t["body"])) for t in TICKETS]
openers = Counter(t["body"].split()[0] for t in TICKETS)
tokens = [w for t in TICKETS for w in re.findall(r"[a-z']+", t["body"].lower())]
messy = sum(bool(re.search(r"Traceback|Exception|[{}\[\]<>]|[A-Z]{5,}", t["body"])) for t in TICKETS)

print(f"mean sentence length per ticket: {min(mean_lengths):.0f} to {max(mean_lengths):.0f} words "
      f"(standard deviation across tickets {statistics.pstdev(mean_lengths):.1f})")
print(f"first word of each ticket: {dict(openers.most_common())}")
print(f"distinct words / total words across all tickets: {len(set(tokens))} / {len(tokens)} "
      f"= {len(set(tokens)) / len(tokens):.2f}")
print(f"tickets containing pasted code, logs, brackets or shouting: {messy} of {len(TICKETS)}")

mean sentence length per ticket: 11 to 21 words (standard deviation across tickets 3.0)
first word of each ticket: {"We're": 4, 'Our': 2, 'We': 2, 'On': 1, 'Since': 1, 'A': 1, 'Roughly': 1}
distinct words / total words across all tickets: 406 / 928 = 0.44
tickets containing pasted code, logs, brackets or shouting: 0 of 12


Mean sentence length runs from 11 to 21 words per ticket. The standard deviation across tickets is 3.0. Eight of twelve tickets open with "We're", "Our" or "We". There are 406 distinct words across 928. Zero tickets contain pasted code, log lines, brackets or shouting. Real inbound support text has all of those. It also has non-native English, one-line tickets, page-long tickets, and the occasional live key. Any metric that scores tone, clarity or professionalism here is scoring our prose. It is not scoring a population of developers. We will not read its numbers as saying anything about production.

Three more limits cannot be measured from inside the notebook. We put them on the record here. The severity labels are our reading of the SLA's letter. Take the key-rotation ticket, T-1007. We labelled it P3, because rotation is working as documented and nothing is failing. In another run of these cells the classifier called it P2. The SLA has no category for security questions. So that label is a decision of ours, not a fact. The money check is a phrase pattern, not a judgement. A summary that said "overbilled" would count as having lost the money. And every rate above belongs to one model, one prompt and one date. The same corpus under a different summarizer would give a different pattern. That is what we built it for.

## What teams get wrong

**Generating the corpus from the prompt the application uses.** Suppose we had asked the same model to "write a support ticket about a duplicate charge". The duplicate charge would have been the subject line every time. The summarizer would have kept it every time. The trap in this post could not exist. The model's picture of a ticket is the assistant's picture of a ticket. We had to write the four money tickets against that picture, the way developers actually write. Their problem first. Ours in passing.

**Writing only the happy path.** Twelve tickets like T-1002 would pass on every commit. T-1002 is a clean rate-limit question with one page that answers it. A suite of those tells us nothing. The controls belong in the suite. They are not the suite. Most of the misses in the full run above came from tickets we wrote to produce one. One came from a control, T-1004. A control that fails is a different kind of news. It is not a trap firing. It is the baseline moving. That is exactly what the controls are there to show.

**Never documenting why each case exists.** The `reason` field on each golden is a single sentence. Without it, T-1010 is a repeat of the webhooks theme. T-1007 is a dull question with a dull answer. T-1009 is a money ticket that never fails. All three are candidates for deletion by whoever next wants a faster suite. With it, each is a deliberate probe of a specific failure. In our experience, the traps that get pruned are the ones nobody wrote down.

**Treating the corpus as fixed forever.** The fingerprint is a version, not a freeze. Production will produce ticket shapes this set does not have. They belong in it. The rule that keeps that safe is the one we followed in this post. Add, never edit. Pages and tickets keep their names and fields. New cases are appended. The fingerprint changes. Every later notebook's assertion then tells us which results predate the change. A corpus that cannot grow stops matching production. One that grows by silent edits stops matching its own history.

In [14]:
tokens_in = sum(u.get("input_tokens", 0) for u in USAGE)
tokens_out = sum(u.get("output_tokens", 0) for u in USAGE)
thinking = sum((u.get("output_token_details") or {}).get("reasoning", 0) for u in USAGE)
cost = tokens_in * PRICE_IN + tokens_out * PRICE_OUT

print(f"model calls      {len(USAGE)}   (judge calls: 0, no metric runs in this post)")
print(f"tokens in / out  {tokens_in} / {tokens_out}   (thinking tokens inside 'out': {thinking})")
print(f"cost             ${cost:.4f} at gemini-3.6-flash standard pricing")
print(f"wall clock       {time.time() - T0:.0f} s since the setup cell")

model calls      34   (judge calls: 0, no metric runs in this post)
tokens in / out  7739 / 24116   (thinking tokens inside 'out': 23558)
cost             $0.0962 at gemini-3.6-flash standard pricing
wall clock       72 s since the setup cell


Thirty-four model calls. None of them a judge. 7,739 tokens in and 24,116 out. Of the output, 23,558 are thinking tokens. Those are the tokens the model spends reasoning before it answers. Google bills them as output. So almost the whole output bill went on reasoning ahead of a 25-word sentence or a two-character severity. At gemini-3.6-flash standard pricing the run cost $0.0962. It took 72 seconds from the setup cell to here. We will come back to the thinking share when later posts multiply judge calls per test case. At this model's default thinking level, the output side dominates cost even when the visible output is one sentence. A cheaper model or a lower thinking level moves the bill far more than it moves the tokens we can see. Every later post ends with a ledger line like this one. That lets us compare call counts and costs across the series.

## The setup cell every later post opens with

We built everything above in pieces so that we could explain it. Later posts re-create the whole scaffold in one cell. They point back here in a sentence and get on with the metric. This is that cell. The data is laid out one page and one ticket per line. That keeps the cell short enough to scroll past. The readable versions are in the sections above. The last three lines prove the two are the same. They recompute the fingerprint and assert it against the value printed in the tickets section. If a later post edits a ticket by accident, the assertion fails before a single model call is made.

In [15]:
# Shared Halcyon scaffolding, built in post 0 and re-created verbatim in every later post.
# !uv pip install deepeval==4.2.1 langgraph==1.2.11 langchain-core==1.6.2 langchain-google-genai==4.4.0 google-genai==2.22.0
import os, re, json, time, hashlib, logging, statistics
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version
from typing import TypedDict

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from deepeval.dataset import Golden

logging.getLogger("google_genai.models").setLevel(logging.ERROR)   # silence an SDK advisory about tool calling

if not os.environ.get("GEMINI_API_KEY"):
    raise RuntimeError("Set GEMINI_API_KEY in the environment before running this notebook.")

JUDGE_MODEL = "gemini-3.6-flash"      # the judge every later post uses; this post never calls it
APP_MODEL = "gemini-3.6-flash"        # the model inside the assistant under test
MAX_CONCURRENCY = 4                   # model calls in flight at any moment
PRICE_IN, PRICE_OUT = 0.75 / 1e6, 3.75 / 1e6   # USD per token, gemini-3.6-flash standard tier, 2026 rate
T0 = time.time()
USAGE = []                            # one usage_metadata dict per model call, for the cost ledger


def with_retry(fn, *args, attempts=5, base_delay=2.0):
    """Call fn(*args); on a 429 or 503 wait 2, 4, 8, 16 seconds and try again."""
    for attempt in range(attempts):
        try:
            return fn(*args)
        except Exception as exc:
            transient = any(code in str(exc) for code in ("429", "503", "RESOURCE_EXHAUSTED"))
            if not transient or attempt == attempts - 1:
                raise
            time.sleep(base_delay * 2 ** attempt)


# No temperature, top_p or top_k: Google lists them as deprecated sampling parameters to strip from requests.
llm = ChatGoogleGenerativeAI(model=APP_MODEL, google_api_key=os.environ["GEMINI_API_KEY"], max_retries=0)

DOCS = [dict(zip(('slug', 'title', 'status', 'updated', 'body'), row)) for row in [
    ("quickstart", "Quickstart", "current", "2026-03-01", "Create an account, then generate a test API key from the dashboard; test keys start with sk_test_ and live keys with sk_live_. Install the SDK with pip install halcyon (the current SDK is 3.x). Create your first charge with POST /v1/charges, passing amount in minor units (an integer: 1999 means 19.99), a lowercase ISO 4217 currency such as usd or eur, and a source token from Halcyon.js. Send the Halcyon-Version header with the API version you tested against so later changes do not reach you until you opt in. Use test card 4242 4242 4242 4242 for a successful payment and 4000 0000 0000 0002 for a decline."),
    ("api-conventions", "API conventions", "current", "2026-03-01", "The API is served over HTTPS at https://api.halcyon.dev and is date-versioned: send Halcyon-Version: 2026-03-01 (or an earlier version) with each request; requests without the header use the account's default version. Amounts are integers in minor units and currencies are lowercase ISO 4217 codes. Every POST to a money-moving endpoint (charges, refunds, payouts) must include an Idempotency-Key header, a client-generated unique string; a UUID v4 is recommended. Replaying a request with the same key within 24 hours returns the original response instead of moving money again. Requests without the header are accepted for backward compatibility, so a retried POST without one creates a second charge. Keys are scoped to the account and to the endpoint."),
    ("authentication", "Authentication", "current", "2026-01-15", "Authenticate every request with an Authorization: Bearer header carrying your secret key. Test-mode keys (sk_test_) never move real money; live-mode keys (sk_live_) do. Restricted keys can be limited to read-only access or to specific resources. Rotate a key from the dashboard: the new key is active immediately and the old key keeps working for 24 hours so deployments can roll over without downtime; revoke it sooner from the same screen if it was exposed. A missing or invalid key returns HTTP 401 with type authentication_error. Never send secret keys from browsers or mobile apps; use Halcyon.js publishable keys (pk_) there."),
    ("errors", "Errors", "current", "2026-03-01", "Errors return a JSON body with type, code, message and request_id. Types map to HTTP status: invalid_request_error (400), authentication_error (401), card_error (402), rate_limit_error (429) and api_error (500 to 504). card_error codes include card_declined, insufficient_funds, expired_card and incorrect_cvc; show the cardholder a generic message and let them try another card. Requests that return 5xx may have partially completed: retry them with exponential backoff, and quote the request_id when you contact support. Do not retry 4xx errors other than 429."),
    ("rate-limits", "Rate limits", "current", "2025-11-01", "Live mode allows 100 requests per second per account with short bursts to 200; test mode allows 25 requests per second. Over the limit, requests fail with HTTP 429, type rate_limit_error, and a Retry-After header giving the number of seconds to wait. Back off exponentially with jitter rather than retrying in a tight loop, and spread bulk work such as migrations or invoice runs over time or through a queue. Webhook deliveries to your endpoint do not count against the limit. Contact support with your expected peak if you need a higher limit."),
    ("charges-create", "Create a charge", "current", "2026-03-01", "POST /v1/charges creates a charge. Required parameters: amount (integer, minor units), currency (lowercase ISO 4217) and source (a token from Halcyon.js or a saved payment method id starting pm_). Optional: description, metadata (up to 20 key-value pairs), and capture, which defaults to true; pass capture=false to authorize only, then capture within 7 days with POST /v1/charges/{id}/capture. The response is a charge object with id (ch_), status (succeeded, pending or failed), amount, currency, amount_refunded and failure_code. Card declines return HTTP 402 with a card_error body; a charge that could not be attempted at all returns 400."),
    ("refunds", "Refunds", "current", "2026-02-10", "POST /v1/refunds with charge (a ch_ id) refunds a charge in full; pass amount to refund part of it. A charge can be refunded several times until the refunded total reaches the original amount; a request beyond the remaining balance returns 400 with code amount_exceeds_refundable. Refunds of a pending charge are rejected until the charge succeeds. The refund object has id (re_), status (pending, succeeded or failed) and amount. Refunds settle to the cardholder in 5 to 10 business days depending on the issuing bank; status succeeded means Halcyon has released the funds, not that the cardholder has seen them yet."),
    ("webhooks-overview", "Webhooks", "current", "2026-03-01", "Halcyon notifies your endpoint about events by POSTing a JSON event object with id (evt_), type, created and data. Event types include charge.succeeded, charge.failed, refund.succeeded and dispute.opened. Your endpoint must return a 2xx within 10 seconds; otherwise the delivery is retried with exponential backoff (1 minute, 5 minutes, 30 minutes, 2 hours, then every 6 hours) for up to 72 hours, after which the event is marked failed and shown in the dashboard. Delivery is at-least-once: the same event can arrive more than once and events can arrive out of order, so store the event id and ignore repeats. Always verify the signature header before trusting an event; see the webhook signatures pages."),
    ("webhook-signatures-v1", "Webhook signatures (v1)", "deprecated", "2026-03-01", "Each webhook request carries an X-Halcyon-Signature header containing a hex-encoded HMAC-SHA1 of the raw request body, keyed with your endpoint's signing secret (whsec_). To verify, compute HMAC-SHA1 over the exact bytes you received, before any JSON parsing, and compare with the header using a constant-time comparison. Reject the event if the values differ. Do not re-serialize the JSON before hashing, since key ordering and whitespace change the digest. This scheme was deprecated on 2026-03-01 in favour of webhook-signatures-v2, which uses HMAC-SHA256 with a timestamped Halcyon-Signature header; it remains in service for accounts pinned to API versions before 2026-03-01."),
    ("webhook-signatures-v2", "Webhook signatures (v2)", "current", "2026-03-01", "Each webhook request carries a Halcyon-Signature header of the form t=<unix timestamp>,v2=<hex digest>. The signed payload is the timestamp, a period, and the raw request body. To verify, compute HMAC-SHA256 of that payload with your endpoint's signing secret (whsec_), compare with the v2 value using a constant-time comparison, and reject the event if the timestamp is more than 300 seconds old, which defeats replay of captured requests. During a signing secret rotation the header carries two v2 values for 24 hours; accept the event if either matches. This is the default scheme for API version 2026-03-01 and later, and the only scheme SDK 3.x verifies."),
    ("changelog-2026-03", "Changelog: API version 2026-03-01", "current", "2026-03-01", "API version 2026-03-01. Webhook signatures v2 (HMAC-SHA256, timestamped Halcyon-Signature header) are now the default; v1 (HMAC-SHA1, X-Halcyon-Signature) is deprecated and is sent only to accounts pinned to earlier versions. Halcyon SDK 3.0 ships with v2 verification helpers and no longer verifies v1 headers; SDK 2.x continues to verify v1 only. The Idempotency-Key replay window is extended from 1 hour to 24 hours, and reusing a key with a different request body now returns 400 with code idempotency_key_reused instead of silently returning the earlier response. Request and response shapes for charges and refunds are unchanged."),
    ("support-sla", "Support and SLA", "current", "2026-01-15", "Support is available to all accounts through the dashboard and support@halcyon.dev. Tickets are triaged into three priorities. P1: any report that funds have been affected, including duplicate or double charges, charges of the wrong amount, and refunds or payouts that did not arrive, regardless of how the report is worded or whether the ticket is mainly about something else; first response within 1 hour, 24 hours a day. P2: a live-mode integration that is failing, such as authentication errors, webhook deliveries or signature verification failing, or unexpected 4xx or 5xx responses from live endpoints; first response within 4 business hours. P3: how-to questions, documentation gaps, feature requests and anything in test mode; first response within 1 business day."),
]]

TICKETS = [dict(zip(('id', 'submitted', 'subject', 'body'), row)) for row in [
    ("T-1001", "2026-08-03", "Which signature header am I supposed to verify?", "We're integrating webhooks and every event we receive has an X-Halcyon-Signature header but no Halcyon-Signature header, so the verify() helper in SDK 3.1 throws 'no v2 signature present'. The webhook overview says to verify signatures and links to two pages. Our account was created in 2025 and I think we're pinned to 2025-11-01. Is v1 still supported, do we need to move to v2, and how do we change the API version without breaking our existing charge flow? This is blocking our launch on the live account."),
    ("T-1002", "2026-08-04", "429s in test mode when our integration suite runs", "Our CI runs about 40 requests per second against test mode at peak and we get bursts of rate_limit_error with Retry-After: 1. Live mode is fine. Is the test-mode limit documented anywhere, and can it be raised for CI? We can add a throttle but it makes the suite roughly three times slower."),
    ("T-1003", "2026-08-05", "Retry storm after our webhook endpoint returned 500s", "On Thursday a bad deploy made our webhook endpoint return 500 for about forty minutes. Since then we've received roughly 4,000 redeliveries, some for events from days ago, and they are still trickling in. Is there a way to cancel pending retries for an endpoint, or do we just absorb them until the 72 hours are up? Probably unrelated, but while going through the logs we found a few orders from that afternoon that were charged twice, two charge ids each; we assume our order service retried the charge call on timeout and we'll refund them on our side. Mainly we need the retries to stop."),
    ("T-1004", "2026-08-06", "Getting 400 idempotency_key_reused on retries", "Since we moved to API version 2026-03-01 last week, our retry path fails with 400 idempotency_key_reused. We generate one Idempotency-Key per order and reuse it if the first attempt fails, which used to work. Looking closer, our retry adds a metadata.attempt field to the body. Is the body compared exactly? What is the recommended pattern for retries where the body legitimately changes? This is live traffic; failed retries currently land in a dead-letter queue and someone replays them by hand."),
    ("T-1005", "2026-08-07", "Partial refund returns amount_exceeds_refundable", "A customer paid 120.00 EUR and we refunded 80.00 EUR last week. Refunding the remaining 40.00 EUR today returns 400 amount_exceeds_refundable. It turns out our cancellation flow had already issued a separate 40.00 EUR refund that nobody noticed, so the error is correct. What we actually need: is there a field on the charge that tells us the remaining refundable balance, so we can check before calling refunds? And is there any per-month cap on refunds?"),
    ("T-1006", "2026-08-10", "SDK 3.0 upgrade broke webhook verification, rolled back", "We upgraded from SDK 2.9 to 3.0 on Monday. From the first deploy every webhook failed verification with 'no v2 signature present' and our fulfilment queue stalled. We rolled back to 2.9 after about 25 minutes and verification works again. Our account is on API version 2025-11-01. What exactly changed in 3.0, and what is the migration order: SDK first or API version first? One side effect: while the queue was stalled a colleague re-ran the stuck jobs by hand and two customers were charged a second time; we've refunded one and are chasing the other. The verification question is the one we need answered before we try again."),
    ("T-1007", "2026-08-11", "Rotated our live key but the old one still works", "We rotated our live secret key from the dashboard this morning as part of our quarterly rotation. Two hours later, requests using the old key still succeed. Is rotation delayed, or did it not take? We expected the old key to stop working straight away. Also, is there an audit log showing which key made which request?"),
    ("T-1008", "2026-08-12", "How do we simulate a dispute in test mode?", "We're building our dispute.opened handler and can't find a way to trigger a dispute against a test-mode charge. Is there a test card number or a dashboard button that opens a dispute? Also, what fields does the dispute object carry? Not urgent, we're a few weeks from launch."),
    ("T-1009", "2026-08-13", "Batch import script times out around row 200", "We're migrating 1,800 customers from our old provider. The script imports each saved payment method and immediately creates the first invoice charge. It runs fine for about 200 rows and then requests start timing out at 30 seconds; the script retries each failed row up to three times and eventually finishes, but the whole run takes hours. Is there a batch endpoint, or a recommended concurrency for imports? Somewhat related: our finance team says a dozen or so of the imported customers have two invoice charges instead of one, which we think is the retry path. We can clean that up ourselves once the import is stable."),
    ("T-1010", "2026-08-14", "Duplicate webhook deliveries and events arriving out of order", "We're seeing the same evt_ id delivered two or three times, sometimes minutes apart, and occasionally a refund.succeeded arrives before the charge.succeeded for the same order. No money problem that we can see, every charge is correct in the dashboard, but our handler assumed one delivery per event and now double-writes some rows. Is this expected behaviour? Should we be deduplicating on our side and, if so, on which field?"),
    ("T-1011", "2026-08-17", "Intermittent 502s from POST /v1/charges since Tuesday", "Roughly one in fifty POST /v1/charges calls returns a 502 with an HTML body rather than your JSON error format, all from the eu-west region. Our client retries 5xx with backoff as your errors page recommends and the retry almost always succeeds, so checkout is mostly working. Request ids for six failures are attached. Two customers have emailed our support saying their statement shows the same order twice, which I assume is the retry; we'll sort that out with them. The 502s are what we need you to look at."),
    ("T-1012", "2026-08-18", "Do you support Apple Pay and Google Pay?", "Our product team wants wallet payments at checkout. I can't find Apple Pay or Google Pay anywhere in the docs. Is it supported through Halcyon.js, on a roadmap, or would we need a second provider for wallets? Also, is there a fee difference for wallet payments?"),
]]

def corpus_fingerprint(docs, tickets):
    return hashlib.sha256(json.dumps([docs, tickets], sort_keys=True).encode()).hexdigest()[:12]

SUMMARIZE = ("You write one-line summaries for Halcyon's support triage queue. Summarize the ticket below "
             "in one sentence of at most 25 words, in plain English.\n\nTicket:\n{ticket}")

def ticket_text(ticket):
    return f"Subject: {ticket['subject']}\n\n{ticket['body']}"

class AppState(TypedDict):
    question: str
    context: list[str] | None
    answer: str

SYSTEM = ("You are Halcyon's developer support assistant. Halcyon is a payments API vendor. When "
          "documentation pages are supplied, answer from them alone and say so if they do not cover the "
          "question. Be concise and specific.")

def respond(state: AppState) -> dict:
    prompt = state["question"]
    if state.get("context"):
        prompt = "Documentation pages:\n\n" + "\n\n".join(state["context"]) + f"\n\nQuestion: {prompt}"
    reply = with_retry(llm.invoke, [SystemMessage(SYSTEM), HumanMessage(prompt)])
    USAGE.append(reply.usage_metadata or {})
    return {"answer": reply.text.strip()}

builder = StateGraph(AppState)
builder.add_node("respond", respond)
builder.add_edge(START, "respond")
builder.add_edge("respond", END)
APP = builder.compile()

def answer(question, context=None):
    return APP.invoke({"question": question, "context": context})["answer"]

SLA = next(d for d in DOCS if d["slug"] == "support-sla")["body"]
CLASSIFY = ("Assign a support priority to the ticket summary below using Halcyon's SLA. Reply with exactly "
            "one of P1, P2 or P3 and nothing else.\n\nSLA:\n{sla}\n\nTicket summary: {summary}")

def triage(ticket):
    """Ticket in, one-sentence summary plus severity out. The severity step sees only the summary."""
    summary = answer(SUMMARIZE.format(ticket=ticket_text(ticket)))
    verdict = re.search(r"P[123]", answer(CLASSIFY.format(sla=SLA, summary=summary)))
    return {"id": ticket["id"], "summary": summary, "severity": verdict.group(0) if verdict else "unparsed"}

GOLDEN_SPEC = {  # severity, trap, pages, reference summary, why it exists
    "T-1001": ("P2", "two-truths", ["webhook-signatures-v1", "webhook-signatures-v2", "changelog-2026-03", "api-conventions"], "Live webhooks carry only the v1 X-Halcyon-Signature header, so SDK 3.1 verification fails; asks whether to move the account to API version 2026-03-01.", "Both signature pages are published and both are correct for someone; the right answer depends on the account's API version."),
    "T-1002": ("P3", None, ["rate-limits"], "CI hits test-mode rate limits at 40 requests per second; asks whether the limit is documented or can be raised.", "A plain how-to with one page that answers it fully; the control case."),
    "T-1003": ("P1", "money-buried", ["webhooks-overview", "api-conventions"], "Webhook retry storm after endpoint 500s; wants pending retries cancelled and reports several orders charged twice during the incident.", "The ticket is about retries; the double charge is an aside at the end. P1 under the SLA."),
    "T-1004": ("P2", None, ["api-conventions", "changelog-2026-03", "errors"], "Retries reusing an Idempotency-Key fail with 400 idempotency_key_reused on live traffic because the retry body differs; asks for the correct retry pattern.", "Needs two pages combined: the conventions rule and the changelog entry that changed it."),
    "T-1005": ("P3", None, ["refunds", "charges-create"], "Second partial refund rejected with amount_exceeds_refundable; asks how to read the remaining refundable balance before refunding.", "Mentions refunds and amounts without any funds being affected; a P3 that sounds like money."),
    "T-1006": ("P1", "money-buried", ["changelog-2026-03", "webhook-signatures-v2", "webhook-signatures-v1"], "SDK 3.0 upgrade made webhook verification fail and was rolled back; two customers were charged twice during the stall; asks the migration order.", "Leads with a failed upgrade; the double charge is a side effect the developer plays down."),
    "T-1007": ("P3", None, ["authentication"], "After a scheduled key rotation the old live key still works two hours later; asks whether that is expected and for how long.", "Sounds like a security incident and is documented behaviour; tests calm reading of docs."),
    "T-1008": ("P3", "no-coverage", [], "Cannot trigger a dispute against a test-mode charge; asks how to simulate one and what the dispute object contains.", "No page covers this. The correct answer is to say so, not to invent a test card."),
    "T-1009": ("P1", "money-buried", ["rate-limits", "api-conventions", "charges-create"], "Bulk import of 1,800 customers times out after about 200 rows; retries have created duplicate invoice charges for a dozen customers.", "Leads with timeouts and batch advice; the duplicate charges are 'somewhat related'."),
    "T-1010": ("P3", "near-miss", ["webhooks-overview"], "Same webhook event delivered several times and out of order; no money affected; asks whether that is expected and how to deduplicate.", "Uses 'duplicate' and 'double' about deliveries, not charges. A keyword rule fires; the SLA does not."),
    "T-1011": ("P1", "money-buried", ["errors", "charges-create", "api-conventions"], "About one in fifty live POST /v1/charges calls returns 502 from eu-west; retries have charged two customers twice.", "Leads with 502s and follows the errors page's own retry advice; the retry is what double-charged customers."),
    "T-1012": ("P3", "no-coverage", [], "Asks whether Apple Pay and Google Pay are supported through Halcyon.js or planned.", "No page covers wallets. Tests whether the assistant admits a documentation gap."),
}

GOLDENS = []
for t in TICKETS:
    severity, trap, pages, summary, reason = GOLDEN_SPEC[t["id"]]
    GOLDENS.append(Golden(name=t["id"], input=ticket_text(t), expected_output=summary,
                          additional_metadata={"expected_severity": severity, "trap": trap,
                                               "pages": pages, "reason": reason}))

CORPUS_FINGERPRINT = corpus_fingerprint(DOCS, TICKETS)
assert CORPUS_FINGERPRINT == "e4485ba0f187", "DOCS or TICKETS drifted from post 0"
print("corpus fingerprint", CORPUS_FINGERPRINT)

corpus fingerprint e4485ba0f187


## What Halcyon has now, and what it still cannot do

The fingerprint is `e4485ba0f187`, and it matched.

Halcyon now has a documentation corpus of twelve pages with three documented traps. A deprecated page that answers the same question as its replacement. A requirement stated on one page and missing from the page where it is needed. A severity rule that only works if the text it reads is complete. It has twelve tickets. Four are P1 whatever they sound like. Two ask about things no page covers. It has a triage assistant that turns a ticket into a summary and a severity. And it has a golden set. For every ticket, that set records the expected severity, the pages that answer it, a reference summary, and the reason the ticket exists.

What it cannot do is measure anything. The triage run above compared strings and counted phrases. Nobody has scored a summary. The number that matters is how often the assistant's one sentence keeps the fact that makes a ticket P1. Right now that is a pattern match with a known false positive. It is not a metric.

The next post puts the first metric on exactly that step, the summary. It asks whether a metric built to score summaries notices what our pattern match caught here. A later post turns the documentation traps loose on a retrieval-backed version of `answer()`. The deprecated page and the missing header requirement are waiting there.

Three rules we are taking forward. Build the failures into the data before building anything that scores it, and run each trap to see it fire. A trap that has never fired is a hypothesis. Write the reason into the golden, not into memory. Pin the corpus with a fingerprint, grow it by adding, and let the assertion say when history and data have parted.

### Resources

- DeepEval: datasets and goldens: https://deepeval.com/docs/evaluation-datasets
- LangGraph overview: https://docs.langchain.com/oss/python/langgraph/overview
- Gemini 3.6 Flash model page: https://ai.google.dev/gemini-api/docs/models/gemini-3.6-flash
- Gemini API pricing (thinking tokens are billed as output): https://ai.google.dev/gemini-api/docs/pricing
- Latest Gemini models guide, including the removal of sampling parameters: https://ai.google.dev/gemini-api/docs/latest-model
- langchain-google-genai on PyPI: https://pypi.org/project/langchain-google-genai/
- Companion repository with this notebook and its pyproject.toml: [REPO-LINK]